# Logistic Regression Training Lab

## Guided Tutorial and Student Practice

This notebook provides a structured introduction to training and evaluating a **Logistic Regression** model.

The notebook is divided into two main parts:

### Part A — Guided Tutorial
You will follow a complete machine learning workflow:

1. Understand the classification problem.
2. Create and inspect the dataset.
3. Define the features and target.
4. Split the data into training and testing sets.
5. Build a preprocessing and modeling pipeline.
6. Train the model.
7. Generate class predictions and probability estimates.
8. Evaluate the model using classification metrics.
9. Interpret coefficients and decision thresholds.

### Part B — Student Practice
You will complete independent exercises using partially written code cells.

---

## Learning Outcomes

By the end of this lab, you should be able to:

- Identify a binary classification problem.
- Explain the difference between class labels and probability estimates.
- Separate the feature matrix `X` from the target vector `y`.
- Split data into training and testing sets.
- Train a Logistic Regression model using `fit()`.
- Generate class predictions using `predict()`.
- Generate probability estimates using `predict_proba()`.
- Interpret the intercept and coefficients.
- Evaluate a classifier using Accuracy, Precision, Recall, F1-score, Confusion Matrix, and ROC-AUC.
- Change the classification threshold and explain its effect.

## Notebook Structure

### Part A — Guided Tutorial
1. Logistic Regression Overview  
2. Import Libraries  
3. Create the Dataset  
4. Inspect the Dataset  
5. Visualize the Classes  
6. Define `X` and `y`  
7. Split the Data  
8. Build the Pipeline  
9. Train the Model  
10. Predict Classes  
11. Predict Probabilities  
12. Apply a Decision Threshold  
13. Evaluate the Model  
14. Interpret the Confusion Matrix  
15. Plot the ROC Curve  
16. Interpret Coefficients  
17. Predict a New Example  
18. Compare Different Thresholds  

### Part B — Student Practice
Exercises 1–12, including a final classification challenge.

---
# Part A — Guided Tutorial

Follow the cells in order. Read the explanation before running each code cell.

## 1. Logistic Regression in One Picture

Logistic Regression follows this sequence:

$$
X \rightarrow z = \theta_0 + \theta_1x_1 + \cdots + \theta_nx_n
\rightarrow \sigma(z)
\rightarrow P(y=1 \mid X)
\rightarrow \text{Class 0 or Class 1}
$$

The sigmoid function is:

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

- `fit()` learns the intercept and coefficients.
- `predict_proba()` returns probability estimates.
- `predict()` converts probabilities into final class labels.

## 2. Import the Required Libraries

In [ ]:
# Data handling
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Machine learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    ConfusionMatrixDisplay,
    RocCurveDisplay
)

# Make random results reproducible
RANDOM_STATE = 42

## 3. Create an Extended Teaching Dataset

The lecture used a small **student pass prediction** example with:

- `study_hours`
- `attendance`
- `pass`

The dataset below is an expanded synthetic teaching dataset so that we can perform a fair train/test split.

> `pass = 1` means the student passed.  
> `pass = 0` means the student failed.

In [ ]:
# Create a reproducible synthetic dataset
rng = np.random.default_rng(RANDOM_STATE)

n_students = 160

study_hours = rng.uniform(0.5, 10.0, n_students)
attendance = rng.uniform(45, 100, n_students)

# Create an underlying probability of passing.
# Higher study hours and attendance increase the probability.
z = -8.0 + 0.85 * study_hours + 0.075 * attendance
probability_of_pass = 1 / (1 + np.exp(-z))

# Generate binary labels from the probabilities
passed = rng.binomial(1, probability_of_pass)

df = pd.DataFrame({
    "study_hours": np.round(study_hours, 1),
    "attendance": np.round(attendance, 1),
    "pass": passed
})

df.head(10)

### Inspect the Dataset

In [ ]:
# Display the dataset dimensions
print("Dataset shape:", df.shape)

# Display column data types
print("\nData types:")
print(df.dtypes)

# Check missing values
print("\nMissing values:")
print(df.isna().sum())

# Check class distribution
print("\nClass distribution:")
print(df["pass"].value_counts().sort_index())

### Why do we inspect the class distribution?

A classification model may look accurate even when one class dominates the dataset.

For example, if 95% of students passed, a weak model could predict `pass = 1` for everyone and still obtain 95% Accuracy. Therefore, we inspect both classes before training.

## 4. Visualize the Two Classes

In [ ]:
plt.figure(figsize=(8, 5))

# Plot each class separately using default matplotlib colors
for class_value in sorted(df["pass"].unique()):
    subset = df[df["pass"] == class_value]
    plt.scatter(
        subset["study_hours"],
        subset["attendance"],
        label=f"Class {class_value}",
        alpha=0.75
    )

plt.xlabel("Study Hours")
plt.ylabel("Attendance")
plt.title("Student Pass Classification Dataset")
plt.legend()
plt.show()

## 5. Define the Features `X` and the Target `y`

The model uses the features to predict the target.

- `X`: input features.
- `y`: correct binary class label.

In [ ]:
# Feature matrix
X = df[["study_hours", "attendance"]]

# Target vector
y = df["pass"]

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

## 6. Split the Data

We train the model on one portion of the data and evaluate it on unseen test data.

`stratify=y` keeps approximately the same class proportion in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTesting class distribution:")
print(y_test.value_counts(normalize=True).sort_index())

## 7. Build a Training Pipeline

The pipeline performs two steps:

1. `StandardScaler()` scales the numerical features.
2. `LogisticRegression()` learns the classifier.

Scaling is useful because the two features have different numerical ranges:

- `study_hours` is roughly between 0 and 10.
- `attendance` is roughly between 45 and 100.

In [ ]:
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression())
])

model

## 8. Train the Model Using `fit()`

In [ ]:
# fit() learns the scaler statistics, intercept, and coefficients
model.fit(X_train, y_train)

print("The model has been trained.")

### What happens inside `fit()`?

The model repeatedly:

1. Computes the linear score \(z\).
2. Applies the sigmoid function.
3. Produces probabilities.
4. Compares probabilities with the true labels.
5. Calculates Binary Cross-Entropy / Log Loss.
6. Adjusts the parameters to reduce the loss.

## 9. Predict Class Labels

In [ ]:
# predict() returns the final class label: 0 or 1
y_pred = model.predict(X_test)

comparison = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "Predicted": y_pred
})

comparison.head(10)

## 10. Predict Probabilities

In [ ]:
# predict_proba() returns one probability for each class
y_probability_all_classes = model.predict_proba(X_test)

# Column 0: probability of Class 0
# Column 1: probability of Class 1
y_probability_class_1 = y_probability_all_classes[:, 1]

probability_table = pd.DataFrame({
    "Actual": y_test.to_numpy(),
    "P(Class 0)": y_probability_all_classes[:, 0],
    "P(Class 1)": y_probability_all_classes[:, 1],
    "Predicted Class": y_pred
})

probability_table.head(10)

### Important Difference

```python
model.predict(X_test)
```

returns the final class labels.

```python
model.predict_proba(X_test)
```

returns the probability estimates before thresholding.

## 11. Manually Apply the Default Threshold

In [ ]:
threshold = 0.5

manual_predictions = (y_probability_class_1 >= threshold).astype(int)

print("Are manual threshold predictions equal to model.predict()?")
print(np.array_equal(manual_predictions, y_pred))

## 12. Evaluate the Model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_probability_class_1)

metrics_table = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Value": [accuracy, precision, recall, f1, roc_auc]
})

metrics_table

### Metric Interpretation

- **Accuracy:** proportion of all correct predictions.
- **Precision:** among predicted positives, how many were actually positive?
- **Recall:** among actual positives, how many did the model detect?
- **F1-score:** balance between Precision and Recall.
- **ROC-AUC:** ability to rank positive cases above negative cases across many thresholds.

## 13. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

ConfusionMatrixDisplay(confusion_matrix=cm).plot()
plt.title("Confusion Matrix")
plt.show()

The matrix contains:

- **True Negative (TN):** actual 0, predicted 0.
- **False Positive (FP):** actual 0, predicted 1.
- **False Negative (FN):** actual 1, predicted 0.
- **True Positive (TP):** actual 1, predicted 1.

## 14. Classification Report

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

## 15. ROC Curve

In [ ]:
RocCurveDisplay.from_predictions(
    y_test,
    y_probability_class_1
)

plt.title("ROC Curve")
plt.show()

## 16. Inspect the Learned Parameters

In [ ]:
# Extract the trained LogisticRegression step from the pipeline
logistic_step = model.named_steps["logistic_regression"]

intercept = logistic_step.intercept_[0]
coefficients = logistic_step.coef_[0]

coefficient_table = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": coefficients
})

print("Intercept:", intercept)
display(coefficient_table)

### Interpreting the Coefficient Signs

Because the model was trained after scaling:

- A positive coefficient pushes the probability of `pass = 1` upward.
- A negative coefficient pushes the probability of `pass = 1` downward.
- Coefficient magnitude reflects the effect in standardized units.

The coefficients affect **log-odds**, not probability directly.

## 17. Predict a New Student

In [ ]:
new_student = pd.DataFrame({
    "study_hours": [6.0],
    "attendance": [80.0]
})

predicted_class = model.predict(new_student)[0]
predicted_probability = model.predict_proba(new_student)[0, 1]

print("Predicted class:", predicted_class)
print("Probability of passing:", round(predicted_probability, 4))

## 18. Change the Classification Threshold

In [ ]:
def evaluate_threshold(y_true, probabilities, threshold):
    '''
    Convert probabilities into class labels using a custom threshold,
    then calculate the main classification metrics.
    '''
    predictions = (probabilities >= threshold).astype(int)

    return {
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1-score": f1_score(y_true, predictions, zero_division=0)
    }


threshold_results = pd.DataFrame([
    evaluate_threshold(y_test, y_probability_class_1, 0.3),
    evaluate_threshold(y_test, y_probability_class_1, 0.5),
    evaluate_threshold(y_test, y_probability_class_1, 0.7)
])

threshold_results

### Threshold Trade-off

- Lower threshold → more positive predictions → Recall often increases.
- Higher threshold → fewer positive predictions → Precision may increase.
- The best threshold depends on the cost of False Positives and False Negatives.

---
# Part B — Student Practice

Complete the following exercises independently.

## Instructions

- Do not modify the guided tutorial cells.
- Replace every `...` with valid Python code.
- Run the notebook from top to bottom.
- Write short interpretations for all reflection questions.
- Do not report a metric without explaining what it means.

## Exercise 1 — Understand the Dataset

Answer the following questions in a Markdown cell:

1. What is the target variable?
2. What does Class 1 mean?
3. Is this a Regression or Classification problem?
4. Why is Accuracy alone sometimes insufficient?

### Your Answer

- Target variable:
- Meaning of Class 1:
- Problem type:
- Why Accuracy may be insufficient:

## Exercise 2 — Create a New Feature Matrix

Create `X_ex2` using both:

- `study_hours`
- `attendance`

Create `y_ex2` using the `pass` column.

Then print their shapes.

In [ ]:
# TODO: Create X_ex2 and y_ex2

X_ex2 = ...
y_ex2 = ...

print("X_ex2 shape:", ...)
print("y_ex2 shape:", ...)

## Exercise 3 — Split the Data

Split `X_ex2` and `y_ex2` into training and testing sets.

Requirements:

- Test size = 20%
- `random_state=42`
- Preserve class balance using `stratify`

In [ ]:
# TODO: Split the data

X_train_ex3, X_test_ex3, y_train_ex3, y_test_ex3 = train_test_split(
    ...,
    ...,
    test_size=...,
    random_state=...,
    stratify=...
)

print("Training rows:", ...)
print("Testing rows:", ...)

## Exercise 4 — Build and Train the Model

Create a pipeline containing:

1. `StandardScaler()`
2. `LogisticRegression()`

Train it using the training data from Exercise 3.

In [ ]:
# TODO: Build and train the pipeline

student_model = Pipeline(steps=[
    ("scaler", ...),
    ("logistic_regression", ...)
])

student_model.fit(..., ...)

print("Model training completed.")

## Exercise 5 — Class Predictions and Probabilities

Using `student_model`:

1. Predict the class labels for `X_test_ex3`.
2. Predict the probability of Class 1.
3. Display the first 10 results in a DataFrame.

In [ ]:
# TODO: Generate predictions and probabilities

student_y_pred = ...
student_y_prob = ...

student_results = pd.DataFrame({
    "Actual": ...,
    "Probability of Class 1": ...,
    "Predicted Class": ...
})

student_results.head(10)

## Exercise 6 — Evaluate the Model

Calculate:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

In [ ]:
# TODO: Calculate classification metrics

student_accuracy = ...
student_precision = ...
student_recall = ...
student_f1 = ...
student_roc_auc = ...

pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"],
    "Value": [
        student_accuracy,
        student_precision,
        student_recall,
        student_f1,
        student_roc_auc
    ]
})

## Exercise 7 — Confusion Matrix Interpretation

1. Calculate the confusion matrix.
2. Extract TN, FP, FN, and TP.
3. Write one sentence interpreting the False Negatives.

In [ ]:
# TODO: Calculate and unpack the confusion matrix

student_cm = ...
tn, fp, fn, tp = ...

print("TN:", tn)
print("FP:", fp)
print("FN:", fn)
print("TP:", tp)

### Your Interpretation

False Negatives mean:

## Exercise 8 — Custom Threshold

Use a threshold of `0.40`.

1. Convert probabilities into class predictions.
2. Calculate Precision and Recall.
3. Compare them with the default threshold of 0.50.

In [ ]:
# TODO: Apply threshold = 0.40

custom_threshold = ...
custom_predictions = (... >= ...).astype(int)

custom_precision = ...
custom_recall = ...

print("Precision at threshold 0.40:", custom_precision)
print("Recall at threshold 0.40:", custom_recall)

### Reflection

When the threshold changed from 0.50 to 0.40:

- Precision:
- Recall:
- Why did this happen?

## Exercise 9 — Interpret the Coefficients

Extract the Logistic Regression coefficients from `student_model`.

Create a table containing:

- Feature name
- Coefficient
- Direction: `"Increases Class 1 probability"` or `"Decreases Class 1 probability"`

In [ ]:
# TODO: Extract and interpret coefficients

trained_logistic = ...
student_coefficients = ...

coefficient_interpretation = pd.DataFrame({
    "Feature": ...,
    "Coefficient": ...,
    "Direction": [
        ...
    ]
})

coefficient_interpretation

## Exercise 10 — Predict New Students

Create a DataFrame for the following students:

| Student | Study Hours | Attendance |
|---|---:|---:|
| A | 2.5 | 58 |
| B | 5.5 | 75 |
| C | 8.0 | 92 |

Predict:

1. Probability of passing.
2. Final class.

In [ ]:
# TODO: Create new students and generate predictions

new_students = pd.DataFrame({
    "study_hours": [...],
    "attendance": [...]
}, index=["A", "B", "C"])

new_probabilities = ...
new_classes = ...

new_students_results = new_students.copy()
new_students_results["Probability of Passing"] = ...
new_students_results["Predicted Class"] = ...

new_students_results

## Exercise 11 — Debugging Task

The following code contains mistakes. Fix them.

```python
wrong_X = df["study_hours", "attendance"]
wrong_y = df[["pass"]]

wrong_model = LogisticRegression
wrong_model.fit(wrong_y, wrong_X)

wrong_predictions = wrong_model.predict_proba(X_test_ex3)
```

Identify at least four mistakes.

In [ ]:
# TODO: Write the corrected code below

correct_X = ...
correct_y = ...

correct_model = ...

correct_model.fit(..., ...)

correct_predictions = ...

## Exercise 12 — Final Challenge: Breast Cancer Classification

Use the built-in Breast Cancer dataset from `scikit-learn`.

Tasks:

1. Load the dataset.
2. Use all numerical features.
3. Split the data with `test_size=0.25`, `random_state=42`, and `stratify=y`.
4. Build a pipeline with `StandardScaler` and `LogisticRegression(max_iter=2000)`.
5. Train the model.
6. Calculate Accuracy, Precision, Recall, F1-score, and ROC-AUC.
7. Display the confusion matrix.
8. Explain which metric is especially important when missing a positive case is costly.

> This is an educational dataset and should not be used for real clinical decisions.

In [ ]:
# TODO: Complete the final challenge

from sklearn.datasets import load_breast_cancer

# 1. Load dataset
cancer = ...

# 2. Create X and y
X_cancer = ...
y_cancer = ...

# 3. Split data
X_train_cancer, X_test_cancer, y_train_cancer, y_test_cancer = train_test_split(
    ...,
    ...,
    test_size=...,
    random_state=...,
    stratify=...
)

# 4. Build pipeline
cancer_model = Pipeline(steps=[
    ("scaler", ...),
    ("logistic_regression", ...)
])

# 5. Train
...

# 6. Predict
cancer_pred = ...
cancer_prob = ...

# 7. Evaluate
cancer_metrics = {
    "Accuracy": ...,
    "Precision": ...,
    "Recall": ...,
    "F1-score": ...,
    "ROC-AUC": ...
}

pd.Series(cancer_metrics)

In [ ]:
# TODO: Display the final challenge confusion matrix

cancer_cm = ...

ConfusionMatrixDisplay(confusion_matrix=cancer_cm).plot()
plt.title("Breast Cancer Classification Confusion Matrix")
plt.show()

### Final Challenge Reflection

1. Which class is treated as the positive class in this dataset?
2. Which metric would you prioritize if False Negatives are dangerous?
3. Did the model perform equally well on both classes?
4. What would you examine before using the model in a real application?

---
# Submission Checklist

Before submitting your notebook, confirm that you have:

- Completed every `TODO` cell.
- Run the notebook from top to bottom without errors.
- Included written interpretations, not only numerical outputs.
- Explained the difference between `predict()` and `predict_proba()`.
- Explained the effect of changing the decision threshold.
- Interpreted at least one model coefficient.
- Interpreted the confusion matrix.
- Completed the final challenge.